# 04 · Señales y rezagos: ¿qué pasa días antes de un huaico?

Pregunta: además de la lluvia del día, ¿hay condiciones (mar, calor aguas arriba, humedad de la selva, caudal, suelo) que cambien **días antes** de un día con huaico en Chosica, Chaclacayo, Santa Eulalia y Ricardo Palma?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un análisis descriptivo con un catálogo de fechas incompleto y con mucho reporte en pocos años. Ante una emergencia llama al 115 (INDECI).

La nota completa con cifras, fuentes, límites y lo que falta está en [`docs/04-senales-y-rezagos.md`](../docs/04-senales-y-rezagos.md). Este notebook se puede reejecutar cuando lleguen más datos.

## Hipótesis, escritas antes de mirar los resultados

| | Hipótesis | Qué se esperaría si fuera cierta |
|---|---|---|
| **H1** | El mar (anomalía y alza de 3, 7 y 14 días, Niño 1+2 y costa de Perú/Lima) **sube días antes** del huaico. | AUC por encima de 0,5 con rezagos de 1 a 14 días, también dentro de cada temporada. |
| **H2** | El calor aguas arriba (anomalía de temperatura y de la isoterma de 0 °C en Ticlio, Santa Eulalia alta y Matucana; nieve) **precede** al huaico o al aumento del caudal. | Anomalía positiva con rezago ≥ 1 día. Si el calor solo aparece negativo el mismo día, es la firma de la tormenta (nubes y frío), no un precursor. |
| **H3** | La humedad de la selva (punto de rocío, nubes) y el gradiente de presión selva-costa **preceden** al huaico. | Punto de rocío de la selva alto con rezago ≥ 1 día. |
| **H4** | El caudal (GloFAS) y la humedad del suelo señalan condiciones previas. | AUC alto con rezago ≥ 1 día (se espera que sea lo más fuerte). |
| **H5** | El efecto del mar es de **régimen lento** (entre temporadas), no un disparador diario. | La señal del mar desaparece al quitar la media de cada temporada y aparece en la correlación entre temporadas. |

**Reglas fijadas de antemano**

- Rezago 0 es el propio día: describe la tormenta, **no** es un precursor. Solo cuenta lo que tenga rezago ≥ 1.
- Una variable en un rezago es *señal* si supera la nula (etiquetas movidas 7 a 25 días dentro de la misma temporada, que conserva el número de eventos y su agrupación) y pasa Benjamini-Hochberg con q = 0,10 sobre todas las pruebas.
- Es *robusta* si además conserva el signo con p < 0,10 en 2009+ y sin 1983, 1998 y 2017.
- Benjamini-Hochberg controla la tasa de falsos descubrimientos: si no hubiera nada, rechazaría algo con probabilidad de a lo más 0,10. Como referencia se muestra además cuántas pruebas pasan p < 0,05 frente a las `0,05 × pruebas` que daría el puro azar.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from illapa_research import evaluacion_diaria as ev_d
from illapa_research import nino
from illapa_research import senales as S
from illapa_research.catalogo import cargar_todos

warnings.filterwarnings("ignore")
DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")
eventos = cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)
D = S.cargar_matriz(DATOS, eventos)
F, y, ts, base = D["F"], D["y"], D["temporada"], D["mask"]
temporadas = nino.cargar_temporadas(DATOS)
print("puntos ERA5:", D["puntos_era5"], "| cajas SST:", D["cajas_sst"], "| último día:", D["ultimo_dia"].date())
print(f"{int(y[base].sum())} días con huaico en {len(np.unique(ts[base]))} temporadas; "
      f"{len(S.episodios_de_dias(list(y.index[(y == 1) & base])))} episodios")
{h: len(S.columnas_de(F, h)) for h in ("H1", "H2", "H3", "H4")}

## 1 · Rezagos 0 a 14 días (todo el período 1984-2026)

AUC de cada variable tal como estaba `k` días antes (0,5 = nada). La mitad inferior (< 0,5) significa que el valor era **menor** de lo normal.

In [ ]:
hip = {h: S.columnas_de(F, h) for h in ("H1", "H2", "H3", "H4")}
todas = list(dict.fromkeys(c for v in hip.values() for c in v))
crudo = S.rezagos_nula(F, y, todas, ts, base, range(0, 15), centrar=False, n_null=1000)
print("pruebas:", crudo.attrs["n_pruebas"], "| señales BH:", crudo.attrs["observados"],
      "| p<0,05:", crudo.attrs["observados_p05"], "frente a", round(crudo.attrs["esperados_p05_por_azar"], 1), "por azar")
def mapa(r, titulo, vars_=None):
    p = r.pivot(index="variable", columns="lag", values="auc")
    if vars_:
        p = p.loc[[v for v in vars_ if v in p.index]]
    s = r.pivot(index="variable", columns="lag", values="senal").reindex(p.index)
    fig, ax = plt.subplots(figsize=(9, max(3, 0.28 * len(p))))
    im = ax.imshow(p.values, cmap="RdBu_r", vmin=0.3, vmax=0.8, aspect="auto")
    ax.set_xticks(range(p.shape[1]), p.columns); ax.set_yticks(range(len(p)), p.index, fontsize=7)
    for i in range(len(p)):
        for j in range(p.shape[1]):
            if s.iloc[i, j]:
                ax.text(j, i, "•", ha="center", va="center", fontsize=8)
    ax.set_title(titulo); ax.set_xlabel("días antes (0 = el día)"); fig.colorbar(im, label="AUC"); plt.show()
clave = [c for c in todas if c in ("sst_nino12_anom", "sst_nino12_alza_7d", "sst_costa_peru_alza_14d", "sst_costa_lima_alza_14d",
         "tic_temp_anom", "sea_temp_anom", "mat_temp_anom", "mat_temp_anom_7d", "tic_cero_anom", "sea_cero_anom",
         "selva_punto_rocio_anom_3d", "gradiente_presion_chosica_selva", "caudal_rimac-chosica_alza_3d",
         "caudal_rimac-chosica_max_7d", "chosica_suelo", "mat_suelo_era5")]
mapa(crudo, "AUC por rezago, variables clave (• = señal tras BH)", clave)

## 2 · Sin la media de cada temporada (¿señal diaria o de régimen?) — H1 y H5

Se resta a cada variable la media de su temporada. Lo que queda es solo el momento dentro de la temporada. Después se compara con la correlación **entre temporadas** (un punto por temporada, para que el agrupamiento de días no infle la evidencia).

In [ ]:
centrado = S.rezagos_nula(F, y, todas, ts, base, range(0, 15), centrar=True, n_null=1000)
print("centrado | señales BH:", centrado.attrs["observados"], "de", centrado.attrs["n_pruebas"],
      "| p<0,05:", centrado.attrs["observados_p05"], "frente a", round(centrado.attrs["esperados_p05_por_azar"], 1), "por azar")
mar = [c for c in hip["H1"]]
comp = pd.DataFrame({
    "AUC crudo (lag 1)": crudo[crudo.lag == 1].set_index("variable").auc,
    "AUC centrado (lag 1)": centrado[centrado.lag == 1].set_index("variable").auc,
}).loc[mar]
comp.round(2)

In [ ]:
regimen = S.regimen_temporada(F, y, [c for c in todas if c in mar or c.startswith(("caudal", "chosica_suelo"))], ts, base)
print("p<0,05:", regimen.attrs["observados_p05"], "frente a", round(regimen.attrs["esperados_p05_por_azar"], 1), "por azar")
regimen.sort_values("p").head(12).round(3)

## 3 · Composites alrededor del inicio de cada episodio

Promedio del puntaje z de la variable desde 14 días antes hasta 3 días después del primer día de cada episodio (días con huaico separados por 3 días o menos cuentan como uno). La banda gris es la nula: cada episodio movido a un día cualquiera de la misma temporada.

In [ ]:
dias_temp = F.index[base]
inicios = [d for d in S.episodios_de_dias(list(y.index[(y == 1) & base]))]
print(len(inicios), "episodios")
elegidas = [c for c in clave if c in F.columns][:12]
n = len(elegidas); cols = 3; filas = int(np.ceil(n / cols))
fig, axs = plt.subplots(filas, cols, figsize=(12, 2.4 * filas), sharex=True)
resultados = {}
for ax, c in zip(np.ravel(axs), elegidas):
    r = S.compuesto(F[c], inicios, lambda d: nino.temporada_de(d.date()), dias_temp, range(-14, 4), n_null=500)
    resultados[c] = r
    ax.fill_between(r.lag, r.nula_p5, r.nula_p95, color="0.85"); ax.plot(r.lag, r.z_medio, "k.-", lw=1)
    ax.axvline(0, color="r", lw=0.5); ax.set_title(c, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame({c: [int(((r.lag <= -1) & (r.z_medio > r.nula_p95)).sum()), int(((r.lag <= -1) & (r.z_medio < r.nula_p5)).sum())]
              for c, r in resultados.items()}, index=["días previos por encima de la banda", "días previos por debajo"]).T

## 4 · Control de multiplicidad y robustez

Son cientos de pruebas (variable × rezago). La tabla resume, por hipótesis, cuántas señales hay frente a las que se esperarían por azar, y cuáles son **robustas** (ver reglas arriba).

In [ ]:
mascaras = S.mascaras_robustez(F, ts, base, temporadas)
rob = S.robustez(F, y, todas, ts, mascaras, lags=range(1, 15), centrar=True, n_null=1000)
print("pruebas:", rob.attrs["n_pruebas"], "| robustas:", int(rob.robusta.sum()))
def hipotesis_de(v):
    return next((h for h in ("H1", "H2", "H3", "H4") if v in hip[h]), "")
rob["hipotesis"] = rob.variable.map(hipotesis_de)
resumen = rob.groupby("hipotesis").agg(pruebas=("lag", "size"), senal_1984=("senal 1984+", "sum"),
                                       robustas=("robusta", "sum")).assign(p05_azar=lambda d: (0.05 * d.pruebas).round(1))
resumen

In [ ]:
rob[rob.robusta][["variable", "lag", "auc 1984+", "auc 2009+", "auc sin 1983, 1998 y 2017",
                  "auc solo temporadas Niño", "auc solo temporadas sin Niño"]].sort_values(["variable", "lag"]).round(2)

## 5 · Lo concurrente frente a lo antecedente

El rezago 0 es el propio día. Estos son los valores a rezago 0 que son **más extremos** que a rezago 1, es decir, la firma de la tormenta en curso y no algo que se pudiera ver antes.

In [ ]:
a0 = centrado[centrado.lag == 0].set_index("variable").auc
a1 = centrado[centrado.lag == 1].set_index("variable").auc
firma = pd.DataFrame({"AUC rezago 0": a0, "AUC rezago 1": a1}).assign(salto=lambda d: (d["AUC rezago 0"] - 0.5).abs() - (d["AUC rezago 1"] - 0.5).abs())
firma.sort_values("salto", ascending=False).head(10).round(2)

## Qué no se puede concluir

- Las señales son asociaciones con un catálogo de fechas incompleto, con mucho reporte en pocos años (2017, 1998, 1983) y sin quebrada en la mayoría de las fechas.
- Un AUC de 0,65 a 0,75 con rezago ≥ 1 día no es una alarma: con unos 100 días de huaico entre 6 400 días de temporada, casi todo lo que supere el azar sigue dando muchas falsas alarmas.
- Se probaron cientos de combinaciones: lo que aquí no pasa la corrección por multiplicidad y la robustez debe leerse como no demostrado.
- Faltan datos si en el encabezado aparecen menos puntos ERA5 o cajas de mar de los seis puntos y tres cajas previstos: los resultados se vuelven a calcular al llegar.